# LeetCode 918 — Maximum Sum Circular Subarray bằng Python

Link đề: https://leetcode.com/problems/maximum-sum-circular-subarray/ · Độ khó: **Medium** · Chủ đề: Array, Divide and Conquer, Dynamic Programming, Queue, Monotonic Queue

Skeleton LeetCode:

```python
class Solution:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên, có thể **âm**). Điểm cốt lõi: đoạn con vòng tròn hoặc **không vòng** (Kadane thường) hoặc **vòng** (đi qua điểm nối cuối–đầu). Đoạn vòng có tổng `= tổng toàn mảng − tổng đoạn con liên tiếp ở giữa`, nên cực đại đoạn vòng ⇔ cực tiểu đoạn con giữa: `total − min_subarray`. Phải xử lý riêng trường hợp **toàn bộ số âm** (khi đó `total − min = 0` ứng với đoạn rỗng, không hợp lệ). Oracle: **vét cạn mọi đoạn vòng** `(start, length ≤ n)` với chỉ số `% n`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng vòng (phần tử cuối nối với phần tử đầu); đoạn con khác rỗng, mỗi phần tử dùng **tối đa một lần**; tìm tổng lớn nhất.
2. Nắm công cụ Python: `itertools.accumulate`, thuật toán **Kadane** (cực đại và cực tiểu), `collections.deque` làm hàng đợi đơn điệu, tổng tiền tố/hậu tố.
3. Đi từ **vét cạn mọi đoạn vòng** → **Kadane hai chiều (`total − min`)** → **tiền tố trên mảng nhân đôi + hàng đợi đơn điệu** → **tiền tố cực đại + hậu tố cực đại**.
4. Hiểu mấu chốt: đoạn vòng đi qua mối nối là **phần bù** của một đoạn thường (liên tiếp, không vòng) nên `max_circular = total − min_normal`; nếu `min_normal = total` (toàn bộ mảng là đoạn nhỏ nhất, tức mọi số `< 0`) thì đoạn vòng rỗng, phải bỏ.
5. Tránh lỗi kinh điển: bỏ qua trường hợp toàn âm (trả `0`), chỉ làm Kadane thường, cho phép đoạn dài hơn `n` (lặp phần tử), nhầm `total − min` khi `min` là toàn bộ mảng, và nhầm đoạn vòng với "mảng nhân đôi bất kỳ".

## 2. Đề bài & ví dụ

Cho mảng số nguyên **vòng** `nums` độ dài `n`: phần tử kế tiếp của `nums[i]` là `nums[(i + 1) % n]`, phần tử trước là `nums[(i − 1 + n) % n]`.

Trả về **tổng lớn nhất** của một đoạn con **khác rỗng** của `nums`.

Đoạn con có thể "vòng qua mối nối" nhưng chỉ dùng mỗi phần tử của bộ đệm **tối đa một lần** (độ dài `≤ n`).

**Ví dụ 1:** `nums = [1,-2,3,-2]` → `3` (đoạn `[3]`).

**Ví dụ 2:** `nums = [5,-3,5]` → `10` (đoạn `[5,5]` vòng qua mối nối).

**Ví dụ 3:** `nums = [-3,-2,-3]` → `-2` (đoạn `[-2]`).

**Ràng buộc:** `1 <= n <= 3·10^4`; `-3·10^4 <= nums[i] <= 3·10^4`.

In [1]:
import itertools
import random
import time
from collections import deque
from itertools import accumulate
from typing import List

examples = [([1, -2, 3, -2], 3), ([5, -3, 5], 10), ([-3, -2, -3], -2)]
for a, want in examples:
    print(f"nums = {a} -> {want}")

nums = [1, -2, 3, -2] -> 3
nums = [5, -3, 5] -> 10
nums = [-3, -2, -3] -> -2


## 3. Kiến thức nền

### 3.1 Đoạn con vòng tròn

Đoạn vòng bắt đầu ở `start`, dài `len ∈ [1, n]`: các phần tử `nums[(start + k) % n]`, `k = 0..len−1`. Có hai loại:

- **không vòng**: `start + len ≤ n` (liên tiếp trong mảng thường);
- **vòng**: `start + len > n` (đi qua mối nối): gồm một **hậu tố** và một **tiền tố** (không giao nhau vì `len ≤ n`).

In [2]:
nums = [5, -3, 5]
n = len(nums)
for start in range(n):
    for ln in range(1, n + 1):
        seg = [nums[(start + k) % n] for k in range(ln)]
        kind = "vòng" if start + ln > n else "thường"
        print(f"start={start} len={ln}: {seg} tổng {sum(seg)} ({kind})")

start=0 len=1: [5] tổng 5 (thường)
start=0 len=2: [5, -3] tổng 2 (thường)
start=0 len=3: [5, -3, 5] tổng 7 (thường)
start=1 len=1: [-3] tổng -3 (thường)
start=1 len=2: [-3, 5] tổng 2 (thường)
start=1 len=3: [-3, 5, 5] tổng 7 (vòng)
start=2 len=1: [5] tổng 5 (thường)
start=2 len=2: [5, 5] tổng 10 (vòng)
start=2 len=3: [5, 5, -3] tổng 7 (vòng)


### 3.2 Kadane — đoạn con thường có tổng lớn nhất/nhỏ nhất

`best_end_here = max(x, best_end_here + x)`; đáp án `max` qua mọi vị trí. Tương tự cực tiểu: `min(x, cur + x)`. Cả hai `O(n)`.

In [3]:
def kadane_max(a):
    best = cur = a[0]
    for x in a[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


def kadane_min(a):
    best = cur = a[0]
    for x in a[1:]:
        cur = min(x, cur + x)
        best = min(best, cur)
    return best


print(kadane_max([1, -2, 3, -2]), kadane_min([1, -2, 3, -2]), kadane_max([-3, -2, -3]), kadane_min([5, -3, 5]))

3 -2 -2 -3


### 3.3 Phần bù của đoạn vòng

Đoạn vòng đi qua mối nối gồm hậu tố + tiền tố; phần **không** thuộc nó là một đoạn con liên tiếp **ở giữa** (có thể rỗng khi đoạn vòng là cả mảng). Do đó `tổng đoạn vòng = total − tổng đoạn giữa`. Cực đại hoá đoạn vòng ⇔ cực tiểu hoá đoạn giữa (khác rỗng, vì đoạn vòng không phải toàn mảng khi đoạn giữa không rỗng).

### 3.4 Hàng đợi đơn điệu (monotonic deque)

Để tìm `min` trong **cửa sổ trượt** `O(1)` gộp, giữ `deque` các chỉ số với giá trị tăng dần; đầu hàng là min. Ta dùng ở Cách 3 trên tổng tiền tố.

## 4. Ý tưởng nền tảng / trực giác

`nums = [5, -3, 5]`, `total = 7`.

- Kadane thường (không vòng): `max = 5` (hoặc `5 + (−3) + 5 = 7`): `7`.
- Đoạn con nhỏ nhất (thường): `[-3]`, `min = −3`. Phần bù (vòng): `total − (−3) = 10` = `5 + 5` ✓ (đoạn vòng `[5, 5]`: phần tử cuối rồi phần tử đầu).

Đáp án `max(7, 10) = 10`.

Trường hợp đặc biệt `nums = [-3, -2, -3]`: `total = −8`, `min_normal = −8` (cả mảng). `total − min = 0` ứng với đoạn rỗng: **không hợp lệ**; đáp án là Kadane thường `−2`.

In [4]:
for a in ([5, -3, 5], [1, -2, 3, -2], [-3, -2, -3]):
    total, mx, mn = sum(a), kadane_max(a), kadane_min(a)
    circ = total - mn
    print(f"{a}: total={total}, kadane_max={mx}, kadane_min={mn}, total-min={circ}", "(toàn âm: bỏ)" if mx < 0 else "")

[5, -3, 5]: total=7, kadane_max=7, kadane_min=-3, total-min=10 
[1, -2, 3, -2]: total=0, kadane_max=3, kadane_min=-2, total-min=2 
[-3, -2, -3]: total=-8, kadane_max=-2, kadane_min=-8, total-min=0 (toàn âm: bỏ)


## 5. Cách 1 — Brute Force: mọi đoạn vòng

**Ý tưởng:** với mỗi điểm bắt đầu `start` và mỗi độ dài `ln = 1..n`, cộng `nums[(start + k) % n]`; lấy cực đại. Dùng tổng chạy: với `start` cố định tăng dần độ dài, `O(1)` mỗi bước.

In [5]:
def circular_brute(nums):
    n = len(nums)
    best = nums[0]
    for start in range(n):
        cur = 0
        for ln in range(n):                              # độ dài 1..n: các phần tử nums[(start + ln) % n]
            cur += nums[(start + ln) % n]
            best = max(best, cur)
    return best


for a, want in examples:
    assert circular_brute(a) == want
print([circular_brute(a) for a, _ in examples])

[3, 10, -2]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng O(n^2): 9·10^8 phép ở n = 3·10^4, TLE)
class SolutionBrute:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        n = len(nums)
        best = nums[0]
        for start in range(n):
            cur = 0
            for ln in range(n):
                cur += nums[(start + ln) % n]
                best = max(best, cur)
        return best

**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ.

**Nút thắt:** ta thử mọi điểm bắt đầu; đoạn vòng có cấu trúc "phần bù của một đoạn thường" nên có thể quy về hai bài Kadane.

## 6. Ý tưởng thuật toán / bước đột phá

`answer = max( kadane_max, total − kadane_min )`, trừ khi `kadane_max < 0` (mọi số âm) thì `answer = kadane_max`. Lý do: đoạn thường có tổng lớn nhất là `kadane_max`; đoạn vòng có tổng lớn nhất là `total − (tổng nhỏ nhất của đoạn giữa không rỗng)`. Nếu mọi số đều âm thì đoạn nhỏ nhất là cả mảng và `total − min = 0` (đoạn vòng rỗng) — bỏ.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** Kadane max + `total − Kadane min`, **Cách 3** tiền tố trên mảng nhân đôi + hàng đợi đơn điệu, **Cách 4** tiền tố cực đại + hậu tố cực đại.

### Cách 2 — Kadane hai chiều: `max(max_sub, total − min_sub)`

**Giải thích từng đoạn:**

1. Một lượt tính đồng thời `total`, `max_sub` (Kadane max), `min_sub` (Kadane min).
2. Nếu `max_sub < 0` (mọi phần tử âm) → trả `max_sub`.
3. Ngược lại trả `max(max_sub, total − min_sub)`.

In [7]:
def circular_kadane(nums):
    total = 0
    cur_max = cur_min = 0
    best_max, best_min = nums[0], nums[0]
    for x in nums:
        total += x
        cur_max = max(x, cur_max + x)
        best_max = max(best_max, cur_max)
        cur_min = min(x, cur_min + x)
        best_min = min(best_min, cur_min)
    if best_max < 0:                                     # mọi số âm: total - best_min sẽ là 0 (đoạn rỗng) -> không hợp lệ
        return best_max
    return max(best_max, total - best_min)


for a, want in examples:
    assert circular_kadane(a) == want
print([circular_kadane(a) for a, _ in examples])

[3, 10, -2]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionKadane:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        total = cur_max = cur_min = 0
        best_max = best_min = nums[0]
        for x in nums:
            total += x
            cur_max = max(x, cur_max + x)
            best_max = max(best_max, cur_max)
            cur_min = min(x, cur_min + x)
            best_min = min(best_min, cur_min)
        return best_max if best_max < 0 else max(best_max, total - best_min)

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Tiền tố trên mảng nhân đôi + hàng đợi đơn điệu

**Ý tưởng:** nối đôi `b = nums + nums`, tiền tố `P[0..2n]`. Đoạn `b[i..j−1]` có tổng `P[j] − P[i]`, hợp lệ (độ dài `≤ n`) khi `1 ≤ j − i ≤ n`. Với mỗi `j = 1..2n` cần **min** `P[i]` trong cửa sổ `i ∈ [j − n, j − 1]` — dùng hàng đợi đơn điệu (giá trị tăng dần). `ans = max_j (P[j] − min P[i])`.

In [9]:
def circular_deque(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums + nums))              # tiền tố của mảng nhân đôi, P[0..2n]
    dq = deque()                                         # chỉ số i với P[i] tăng dần; đầu hàng là min trong cửa sổ
    best = float("-inf")
    for j in range(1, 2 * n + 1):
        i = j - 1                                        # thêm ứng viên điểm đầu i = j - 1
        while dq and P[dq[-1]] >= P[i]:
            dq.pop()
        dq.append(i)
        while dq[0] < j - n:                             # đoạn dài tối đa n: điểm đầu i >= j - n
            dq.popleft()
        best = max(best, P[j] - P[dq[0]])
    return best


for a, want in examples:
    assert circular_deque(a) == want
print([circular_deque(a) for a, _ in examples])

[3, 10, -2]


In [10]:
# Code nộp LeetCode cho Cách 3
from collections import deque
from itertools import accumulate


class SolutionDeque:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        n = len(nums)
        P = [0] + list(accumulate(nums + nums))
        dq, best = deque(), float("-inf")
        for j in range(1, 2 * n + 1):
            i = j - 1
            while dq and P[dq[-1]] >= P[i]:
                dq.pop()
            dq.append(i)
            while dq[0] < j - n:
                dq.popleft()
            best = max(best, P[j] - P[dq[0]])
        return best

**Độ phức tạp:** `O(n)` thời gian (mỗi chỉ số vào/ra hàng đợi một lần), `O(n)` bộ nhớ. Không cần phân biệt toàn âm: độ dài `≥ 1` được bảo đảm vì cửa sổ chứa `i = j − 1`.

### Cách 4 — Tiền tố cực đại + hậu tố cực đại

**Ý tưởng:** đoạn vòng = hậu tố bắt đầu `j` + tiền tố kết thúc `i` với `i < j`. Gọi `pre[i] = nums[0] + … + nums[i]`, `suf[j] = nums[j] + … + nums[n−1]`, `best_pre[i] = max_{k ≤ i} pre[k]`. Đoạn vòng tốt nhất `= max_{j ≥ 1} (suf[j] + best_pre[j − 1])`. Đáp án `= max(Kadane thường, đoạn vòng tốt nhất)`.

In [11]:
def circular_prefix_suffix(nums):
    n = len(nums)
    best_pre = [0] * n                                   # best_pre[i] = max tổng tiền tố độ dài 1..i+1
    run, best = 0, float("-inf")
    for i, x in enumerate(nums):
        run += x
        best = max(best, run)
        best_pre[i] = best
    best_wrap = float("-inf")
    suf = 0
    for j in range(n - 1, 0, -1):                        # hậu tố bắt đầu j >= 1, ghép với tiền tố kết thúc tại <= j-1
        suf += nums[j]
        best_wrap = max(best_wrap, suf + best_pre[j - 1])
    return max(kadane_max(nums), best_wrap)


for a, want in examples:
    assert circular_prefix_suffix(a) == want
print([circular_prefix_suffix(a) for a, _ in examples])

[3, 10, -2]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionPrefixSuffix:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        n = len(nums)
        best_pre, run, best = [0] * n, 0, float("-inf")
        for i, x in enumerate(nums):
            run += x
            best = max(best, run)
            best_pre[i] = best
        kad = cur = nums[0]
        for x in nums[1:]:
            cur = max(x, cur + x)
            kad = max(kad, cur)
        wrap, suf = float("-inf"), 0
        for j in range(n - 1, 0, -1):
            suf += nums[j]
            wrap = max(wrap, suf + best_pre[j - 1])
        return max(kad, wrap)

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Không cần suy luận về "toàn âm": hậu tố và tiền tố luôn khác rỗng.

## 8. TRACE từng bước

**Trace Cách 2** với `nums = [5, -3, 5]`:

| `x` | `total` | `cur_max` | `best_max` | `cur_min` | `best_min` |
|---|---|---|---|---|---|
| 5 | 5 | 5 | 5 | 5 | 5 |
| −3 | 2 | `max(−3, 2) = 2` | 5 | `min(−3, 2) = −3` | −3 |
| 5 | 7 | `max(5, 7) = 7` | 7 | `min(5, 2) = 2` | −3 |

`best_max = 7 ≥ 0` ⇒ `max(7, 7 − (−3)) = 10` ✓. Hàm trace:

In [13]:
def trace_kadane(nums):
    total = cur_max = cur_min = 0
    best_max = best_min = nums[0]
    print(f"nums = {nums}")
    print(f"{'x':>4} {'total':>6} {'cur_max':>8} {'best_max':>9} {'cur_min':>8} {'best_min':>9}")
    for x in nums:
        total += x
        cur_max = max(x, cur_max + x)
        best_max = max(best_max, cur_max)
        cur_min = min(x, cur_min + x)
        best_min = min(best_min, cur_min)
        print(f"{x:>4} {total:>6} {cur_max:>8} {best_max:>9} {cur_min:>8} {best_min:>9}")
    ans = best_max if best_max < 0 else max(best_max, total - best_min)
    print(f"best_max = {best_max}; total - best_min = {total - best_min}; đáp án = {ans}" + ("  (toàn âm)" if best_max < 0 else ""))
    return ans


assert trace_kadane([5, -3, 5]) == 10
print()
assert trace_kadane([-3, -2, -3]) == -2

nums = [5, -3, 5]
   x  total  cur_max  best_max  cur_min  best_min
   5      5        5         5        5         5
  -3      2        2         5       -3        -3
   5      7        7         7        2        -3
best_max = 7; total - best_min = 10; đáp án = 10

nums = [-3, -2, -3]
   x  total  cur_max  best_max  cur_min  best_min
  -3     -3       -3        -3       -3        -3
  -2     -5       -2        -2       -5        -5
  -3     -8       -3        -2       -8        -8
best_max = -2; total - best_min = 0; đáp án = -2  (toàn âm)


**Trace Cách 3** với `[5, -3, 5]`: mảng nhân đôi `[5,-3,5,5,-3,5]`, `P = [0,5,2,7,12,9,14]`. Với `j = 4`: cửa sổ `i ∈ [1, 3]`, `min P[i] = 2` (`i = 2`): `P[4] − 2 = 10` ✓ (đoạn `b[2..3] = [5, 5]`).

## 9. Code nộp LeetCode

Chọn **Cách 2** (Kadane hai chiều, `O(1)` bộ nhớ). Giữ nguyên `class Solution`, `maxSubarraySumCircular(self, nums)`:

In [14]:
class Solution:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        total = cur_max = cur_min = 0
        best_max = best_min = nums[0]
        for x in nums:
            total += x
            cur_max = max(x, cur_max + x)                  # Kadane: đoạn thường tổng lớn nhất kết thúc tại x
            best_max = max(best_max, cur_max)
            cur_min = min(x, cur_min + x)                  # Kadane đối xứng: đoạn thường tổng nhỏ nhất
            best_min = min(best_min, cur_min)
        if best_max < 0:                                   # mọi số âm: phần bù sẽ là đoạn rỗng, không hợp lệ
            return best_max
        return max(best_max, total - best_min)             # không vòng, hoặc vòng = tổng - đoạn giữa nhỏ nhất


print([Solution().maxSubarraySumCircular(a) for a, _ in examples])      # [3, 10, -2]
assert all(Solution().maxSubarraySumCircular(a) == want for a, want in examples)

[3, 10, -2]


## 10. Vì sao đúng

**(a) Phân loại đoạn.** Mỗi đoạn vòng độ dài `ℓ ∈ [1, n]` là đoạn thường (không đi qua mối nối) hoặc đoạn vòng (đi qua mối nối). Đáp án `= max(M₁, M₂)` với `M₁` là tổng lớn nhất của đoạn thường (chính là Kadane) và `M₂` là tổng lớn nhất của đoạn vòng.

**(b) Đoạn vòng là phần bù của một đoạn thường.** Đoạn vòng qua mối nối gồm hậu tố `[j, n−1]` và tiền tố `[0, i]` với `i < j`. Phần còn lại là đoạn thường `mid = [i+1, j−1]` (rỗng khi `j = i + 1`, tức đoạn vòng là cả mảng và đã là đoạn thường). Do đó tổng đoạn vòng `= total − sum(mid)`. Ngược lại, với mọi đoạn thường khác rỗng `mid ≠ cả mảng`, phần bù của nó (hậu tố sau `mid` + tiền tố trước `mid`) là một đoạn vòng hợp lệ (khác rỗng) có tổng `total − sum(mid)`; nếu `mid` chạm một đầu mảng thì phần bù cũng là một đoạn thường, vẫn hợp lệ. Vậy `M₂ = max over mid khác rỗng, mid ≠ cả mảng` của `total − sum(mid)` `= total − (tổng nhỏ nhất của các đoạn thường khác rỗng, khác cả mảng)`.

**(c) Vì sao công thức `max(best_max, total − best_min)` đúng.** Công thức dùng `best_min` là cực tiểu trên **mọi** đoạn thường khác rỗng, kể cả `mid = cả mảng` (cho `total − sum = 0`, ứng với phần bù **rỗng**, không hợp lệ). Xét hai trường hợp:

- `best_max ≥ 0` (có ít nhất một phần tử `≥ 0`): giá trị `0` từ phần bù rỗng không vượt `best_max`, nên việc "lỡ" tính nó không làm đổi `max(best_max, ·)`. Với mọi `mid` khác, `total − sum(mid)` là tổng đoạn hợp lệ. Vậy kết quả đúng bằng `max(M₁, M₂)`.
- `best_max < 0` (mọi phần tử âm): mọi đoạn con khác rỗng có tổng `≤` phần tử lớn nhất của nó, nên đáp án `= max(nums) = best_max`. Công thức `total − best_min` sẽ cho `0` (phần bù rỗng) — sai — nên ta trả riêng `best_max`.

**(d) Cách 3.** Đoạn vòng độ dài `ℓ ≤ n` bắt đầu `start` là `b[start..start+ℓ−1]` trên mảng nhân đôi, tổng `P[start+ℓ] − P[start]`, với `start ∈ [0, n−1]`. Ta xét mọi `i ∈ [j−n, j−1]` (kể cả `i ≥ n`: đoạn ấy trùng với đoạn có `start = i − n`), nên không thiếu và không thừa giá trị.

**(e) Cách 4.** Đoạn vòng thực sự là hậu tố `[j, n−1]` (`j ≥ 1`) cộng tiền tố `[0, i]` với `i ≤ j − 1` (không giao nhau); tổng lớn nhất qua `i` là `best_pre[j−1]`. Đoạn thường lấy từ Kadane. Mọi đoạn thuộc một trong hai loại.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **mọi đoạn vòng** trên mọi mảng nhỏ (mục 12D), và khớp nhau trên mảng lớn (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi đoạn vòng | `O(n²)` | `O(1)` | oracle |
| 2. Kadane hai chiều | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Tiền tố + hàng đợi đơn điệu | `O(n)` | `O(n)` | khái quát cho độ dài tối đa `k` |
| 4. Tiền tố + hậu tố cực đại | `O(n)` | `O(n)` | không cần xử lý toàn âm riêng |

Đo thực tế (`n = 3·10^4`; cách 1 chỉ `n = 2000`):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(918)
n = 30000
big = [rng.randint(-30000, 30000) for _ in range(n)]
for name, fn in [("kadane", circular_kadane), ("deque", circular_deque), ("pre_suf", circular_prefix_suffix)]:
    print(f"n = 3·10^4 {name:8s} {bench(fn, big) * 1e3:8.2f} ms")
print(f"cách 1 với n = 2000: {bench(circular_brute, big[:2000], 1) * 1e3:8.1f} ms (tăng theo n²)")

n = 3·10^4 kadane       4.19 ms
n = 3·10^4 deque       16.52 ms
n = 3·10^4 pre_suf      7.48 ms


cách 1 với n = 2000:    354.6 ms (tăng theo n²)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ với oracle mọi-đoạn-vòng, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(nums):
    """Vét cạn MỌI đoạn vòng: (start, length) với length 1..n, các phần tử nums[(start + k) % n] — dùng slicing trên mảng nhân đôi."""
    n = len(nums)
    doubled = nums + nums
    return max(sum(doubled[s:s + ln]) for s in range(n) for ln in range(1, n + 1))


IMPLS = {
    "brute": circular_brute,
    "kadane": circular_kadane,
    "deque": circular_deque,
    "pre_suf": circular_prefix_suffix,
    "Solution": lambda a: Solution().maxSubarraySumCircular(list(a)),
    "SolutionBrute": lambda a: SolutionBrute().maxSubarraySumCircular(list(a)),
    "SolutionKadane": lambda a: SolutionKadane().maxSubarraySumCircular(list(a)),
    "SolutionDeque": lambda a: SolutionDeque().maxSubarraySumCircular(list(a)),
    "SolutionPrefixSuffix": lambda a: SolutionPrefixSuffix().maxSubarraySumCircular(list(a)),
}


def check(nums, want=None, names=None):
    if want is None:
        want = oracle(nums)
    for name in (names or IMPLS):
        got = IMPLS[name](list(nums))
        assert got == want, (name, nums, got, want)


for a, want in examples:
    assert oracle(a) == want
    check(a, want)
print("ví dụ đề: oracle mọi đoạn vòng và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle mọi đoạn vòng và 9 cài đặt đúng


### B. Edge case

- Một phần tử (dương, âm, `0`).
- Toàn âm: đáp án là phần tử lớn nhất.
- Toàn dương: đáp án là tổng cả mảng.
- Toàn `0`.
- Đoạn tốt nhất **vòng qua mối nối**.
- Đoạn tốt nhất là cả mảng.
- Một số dương nằm giữa các số âm lớn.
- Giá trị cực trị (`±3·10^4`).

In [17]:
cases = [
    ([7], 7), ([-7], -7), ([0], 0), ([-1, -2, -3], -1), ([1, 2, 3], 6), ([0, 0, 0], 0),
    ([5, -3, 5], 10), ([3, -1, 2, -1], 4), ([-2, 4, -5, 4, -5, 9, 4], 15), ([2, -5, 2], 4),
    ([-5, 1, -5], 1), ([30000, -30000, 30000], 60000), ([-30000, -30000], -30000), ([1, -1, 1, -1], 1),
]
for a, want in cases:
    check(a, want)
print(len(cases), "edge case: đúng")

14 edge case: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_all_negative(nums):                            # total - min khi toàn âm cho 0 (đoạn rỗng)
    total = cur_max = cur_min = 0
    best_max = best_min = nums[0]
    for x in nums:
        total += x
        cur_max = max(x, cur_max + x)
        best_max = max(best_max, cur_max)
        cur_min = min(x, cur_min + x)
        best_min = min(best_min, cur_min)
    return max(best_max, total - best_min)             # SAI: thiếu nhánh toàn âm


def bug_no_wrap(nums):                                 # chỉ Kadane thường
    return kadane_max(nums)


def bug_longer_than_n(nums):                           # cho phép độ dài > n (lặp phần tử): mảng nhân đôi với cửa sổ 2n
    n = len(nums)
    doubled = nums + nums
    return max(sum(doubled[s:s + ln]) for s in range(n) for ln in range(1, 2 * n + 1) if s + ln <= 2 * n)


def bug_min_is_max(nums):                              # dùng Kadane max thay cho Kadane min
    return max(kadane_max(nums), sum(nums) - kadane_max(nums))


def bug_min_of_all_prefix(nums):                       # min đoạn giữa chỉ xét tiền tố (không phải đoạn bất kỳ)
    total, run, mn = sum(nums), 0, 0
    for x in nums:
        run += x
        mn = min(mn, run)
    return max(kadane_max(nums), total - mn) if kadane_max(nums) >= 0 else kadane_max(nums)


def bug_wrap_includes_whole_double(nums):              # cộng thêm total vào Kadane thường (đoạn vòng bất kỳ)
    return max(kadane_max(nums), kadane_max(nums) + sum(nums))


BUGS = [bug_all_negative, bug_no_wrap, bug_longer_than_n, bug_min_is_max, bug_min_of_all_prefix, bug_wrap_includes_whole_double]
rng = random.Random(3)
tests = [[rng.randint(-5, 5) for _ in range(rng.randint(1, 7))] for _ in range(800)]
for bug in BUGS:
    bad = next((a for a in tests if bug(a) != oracle(a)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:28s} sai trên {bad}: ra {bug(bad)}, đúng {oracle(bad)}")

bug_all_negative             sai trên [-1]: ra 0, đúng -1
bug_no_wrap                  sai trên [5, 4, -4, 4]: ra 9, đúng 13
bug_longer_than_n            sai trên [4, 3]: ra 14, đúng 7
bug_min_is_max               sai trên [5, 4, -4, 4]: ra 9, đúng 13
bug_min_of_all_prefix        sai trên [5, 4, -4, 4]: ra 9, đúng 13
bug_wrap_includes_whole_double sai trên [4, 3]: ra 14, đúng 7


### D. Vét cạn mọi mảng nhỏ

Mọi mảng độ dài `1..5` với giá trị `−2..2` (`5 + 25 + 125 + 625 + 3 125 = 3 905`) và mọi mảng độ dài `6` với giá trị `{−3, −1, 0, 2}` (`4 096`). Oracle: mọi đoạn vòng.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("brute", "kadane", "deque", "pre_suf", "Solution")
for n, vals in [(1, range(-2, 3)), (2, range(-2, 3)), (3, range(-2, 3)), (4, range(-2, 3)), (5, range(-2, 3)), (6, (-3, -1, 0, 2))]:
    for tup in itertools.product(vals, repeat=n):
        a = list(tup)
        want = oracle(a)
        for name in fast:
            assert IMPLS[name](a) == want, (name, a)
        cnt += 1
print(cnt, "mảng nhỏ: 5 cài đặt khớp oracle mọi đoạn vòng;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 3905 + 4096

8001 mảng nhỏ: 5 cài đặt khớp oracle mọi đoạn vòng; 0.1 s


### E. Ngẫu nhiên có seed và tính chất

Mảng ngẫu nhiên tới `60` (so oracle) và `400` (so Cách 1). Tính chất: (1) **xoay** mảng (dịch vòng) không đổi đáp án; (2) đảo mảng không đổi; (3) `answer ≥ max(nums)` và `answer ≥ kadane_max`; (4) nếu mọi số `≥ 0` thì `answer = total`; (5) nếu mọi số `< 0` thì `answer = max(nums)`.

In [20]:
rng = random.Random(20240918)
for _ in range(2000):
    n = rng.randint(1, 60)
    mode = rng.choice(["mixed", "neg", "pos", "small"])
    if mode == "mixed":
        a = [rng.randint(-30000, 30000) for _ in range(n)]
    elif mode == "neg":
        a = [rng.randint(-30000, -1) for _ in range(n)]
    elif mode == "pos":
        a = [rng.randint(0, 30000) for _ in range(n)]
    else:
        a = [rng.randint(-3, 3) for _ in range(n)]
    want = oracle(a)
    for name in ("kadane", "deque", "pre_suf", "Solution", "SolutionKadane", "SolutionDeque", "SolutionPrefixSuffix"):
        assert IMPLS[name](a) == want, (name, a)
    r = rng.randrange(n)
    assert Solution().maxSubarraySumCircular(a[r:] + a[:r]) == want                    # (1) xoay
    assert Solution().maxSubarraySumCircular(a[::-1]) == want                          # (2) đảo
    assert want >= max(a) and want >= kadane_max(a)                                    # (3)
    if all(x >= 0 for x in a):
        assert want == sum(a)                                                          # (4)
    if all(x < 0 for x in a):
        assert want == max(a)                                                          # (5)
for _ in range(100):
    a = [rng.randint(-30000, 30000) for _ in range(rng.randint(100, 400))]
    assert Solution().maxSubarraySumCircular(a) == circular_brute(a) == circular_deque(a) == circular_prefix_suffix(a)
print("2000 mảng ngẫu nhiên + 100 mảng 100..400 phần tử: khớp; 5 tính chất (xoay, đảo, cận, toàn dương, toàn âm) đúng")

2000 mảng ngẫu nhiên + 100 mảng 100..400 phần tử: khớp; 5 tính chất (xoay, đảo, cận, toàn dương, toàn âm) đúng


### F. Giới hạn: `n = 3·10^4`

Các cách `O(n)` khớp nhau trên mảng lớn nhiều dạng; công thức đóng cho vài dạng.

In [21]:
rng = random.Random(500)
n = 30000
fast = ["kadane", "deque", "pre_suf", "Solution", "SolutionKadane", "SolutionDeque", "SolutionPrefixSuffix"]
datasets = {
    "ngẫu nhiên": [rng.randint(-30000, 30000) for _ in range(n)],
    "toàn dương": [rng.randint(0, 30000) for _ in range(n)],
    "toàn âm": [rng.randint(-30000, -1) for _ in range(n)],
    "xen kẽ +30000/-30000": [30000 if i % 2 == 0 else -30000 for i in range(n)],
    "khối dương hai đầu": [30000] * 10000 + [-30000] * 10000 + [30000] * 10000,
    "toàn 0": [0] * n,
}
for name, a in datasets.items():
    res = {k: IMPLS[k](a) for k in fast}
    assert len(set(res.values())) == 1, (name, res)
    print(f"{name:24s}: đáp án {res['kadane']}")
assert Solution().maxSubarraySumCircular(datasets["toàn dương"]) == sum(datasets["toàn dương"])
assert Solution().maxSubarraySumCircular(datasets["toàn âm"]) == max(datasets["toàn âm"])
assert Solution().maxSubarraySumCircular(datasets["khối dương hai đầu"]) == 600000000
assert Solution().maxSubarraySumCircular(datasets["xen kẽ +30000/-30000"]) == 30000
print("công thức đóng khớp")

ngẫu nhiên              : đáp án 7660562
toàn dương              : đáp án 451958794


toàn âm                 : đáp án -1
xen kẽ +30000/-30000    : đáp án 30000


khối dương hai đầu      : đáp án 600000000
toàn 0                  : đáp án 0
công thức đóng khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên trường hợp toàn âm

In [22]:
print("[-3,-2,-3]:", bug_all_negative([-3, -2, -3]), "| đúng:", Solution().maxSubarraySumCircular([-3, -2, -3]))
assert bug_all_negative([-3, -2, -3]) == 0 and Solution().maxSubarraySumCircular([-3, -2, -3]) == -2

[-3,-2,-3]: 0 | đúng: -2


**Vì sao sai:** khi mọi số âm, `total − min_sub = 0` ứng với đoạn **rỗng**, nhưng đề yêu cầu đoạn **khác rỗng**. **Đúng:** nếu `max_sub < 0` trả `max_sub`.

### Lỗi 2: chỉ làm Kadane thường

In [23]:
print("[5,-3,5]:", bug_no_wrap([5, -3, 5]), "| đúng:", Solution().maxSubarraySumCircular([5, -3, 5]))
assert bug_no_wrap([5, -3, 5]) == 7

[5,-3,5]: 7 | đúng: 10


**Vì sao sai:** đoạn tốt nhất `[5, 5]` đi qua mối nối. **Đúng:** thêm `total − min_sub`.

### Lỗi 3: cho phép độ dài lớn hơn `n`

In [24]:
print("[1,-1,5]:", bug_longer_than_n([1, -1, 5]), "| đúng:", Solution().maxSubarraySumCircular([1, -1, 5]))
bad = next(a for a in tests if bug_longer_than_n(a) != oracle(a))
print("phản ví dụ:", bad, "->", bug_longer_than_n(bad), "| đúng:", oracle(bad))

[1,-1,5]: 10 | đúng: 6
phản ví dụ: [4, 3] -> 14 | đúng: 7


**Vì sao sai:** mỗi phần tử chỉ dùng **tối đa một lần** (độ dài `≤ n`). Nhân đôi mảng không có nghĩa được dùng mọi đoạn của mảng nhân đôi. **Đúng:** hạn chế cửa sổ `≤ n` (hàng đợi đơn điệu) hoặc dùng phần bù.

### Lỗi 4: dùng Kadane max thay vì min cho phần bù

In [25]:
bad = next(a for a in tests if bug_min_is_max(a) != oracle(a))
print(bad, "->", bug_min_is_max(bad), "| đúng:", oracle(bad))

[5, 4, -4, 4] -> 9 | đúng: 13


**Vì sao sai:** phần bù cần **cực tiểu** đoạn giữa để cực đại phần vòng. **Đúng:** `total − kadane_min`.

### Lỗi 5: chỉ xét tiền tố làm đoạn giữa nhỏ nhất

In [26]:
bad = next(a for a in tests if bug_min_of_all_prefix(a) != oracle(a))
print(bad, "->", bug_min_of_all_prefix(bad), "| đúng:", oracle(bad))

[5, 4, -4, 4] -> 9 | đúng: 13


**Vì sao sai:** đoạn giữa có thể bắt đầu ở bất kỳ chỉ số, không chỉ ở đầu mảng. **Đúng:** Kadane cho cực tiểu đoạn con bất kỳ.

### Lỗi 6 (bonus): cộng `total` vào Kadane thường

In [27]:
bad = next(a for a in tests if bug_wrap_includes_whole_double(a) != oracle(a))
print(bad, "->", bug_wrap_includes_whole_double(bad), "| đúng:", oracle(bad))

[4, 3] -> 14 | đúng: 7


**Vì sao sai:** đoạn vòng không phải "đoạn thường cộng cả mảng" (sẽ lặp phần tử). **Đúng:** hậu tố + tiền tố không giao, hoặc phần bù.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đoạn con **vòng tròn**, tổng lớn nhất" → hoặc **không vòng** (Kadane) hoặc **vòng = tổng − đoạn thường nhỏ nhất**; hoặc nhân đôi mảng + tiền tố + hàng đợi đơn điệu cho độ dài `≤ n`.

**Mẹo:**

1. Nghĩ theo phần bù: đoạn vòng = tổng − đoạn giữa.
2. Luôn xử lý riêng trường hợp **toàn âm**.
3. Mảng nhân đôi: giới hạn độ dài `≤ n`.
4. Hàng đợi đơn điệu cho "min/max trong cửa sổ" `O(1)` gộp.
5. Tiền tố + hậu tố cực đại là cách khác, không cần nhánh toàn âm.
6. Oracle: mọi `(start, length)`.

**Họ bài cùng pattern:** Maximum Subarray (53), Maximum Product Subarray (152), Shortest Subarray with Sum at Least K (862), Maximum Sum of Two Non-Overlapping Subarrays (1031), Circular Array Loop (457), Gas Station (134).

## 15. Biến thể

1. **Tổng nhỏ nhất** của đoạn vòng (đối xứng).
2. **Độ dài tối đa `k`** (`k ≤ n`): hàng đợi đơn điệu trên cửa sổ `k`.
3. **Trả về đoạn** (chỉ số bắt đầu và độ dài).
4. **Tích lớn nhất** (khó hơn).
5. **Đoạn vòng có độ dài đúng `k`.**

Cài biến thể 1, 2, 3 và đối chiếu vét cạn:

In [28]:
def min_circular(nums):
    return -circular_kadane([-x for x in nums])


def min_circular_brute(nums):
    n = len(nums)
    d = nums + nums
    return min(sum(d[s:s + ln]) for s in range(n) for ln in range(1, n + 1))


def max_circular_at_most_k(nums, k):
    """Biến thể 2: độ dài đoạn vòng nằm trong [1, k], k <= n; hàng đợi đơn điệu trên tiền tố của mảng nhân đôi."""
    n = len(nums)
    P = [0] + list(accumulate(nums + nums))
    dq, best = deque(), float("-inf")
    for j in range(1, 2 * n + 1):
        i = j - 1
        while dq and P[dq[-1]] >= P[i]:
            dq.pop()
        dq.append(i)
        while dq[0] < j - k:
            dq.popleft()
        best = max(best, P[j] - P[dq[0]])
    return best


def max_circular_at_most_k_brute(nums, k):
    n = len(nums)
    d = nums + nums
    return max(sum(d[s:s + ln]) for s in range(n) for ln in range(1, k + 1))


def best_segment(nums):
    """Biến thể 3: (start, length) của một đoạn vòng tốt nhất (start nhỏ nhất, rồi length nhỏ nhất)."""
    n = len(nums)
    best = oracle(nums)
    d = nums + nums
    for s in range(n):
        for ln in range(1, n + 1):
            if sum(d[s:s + ln]) == best:
                return s, ln


rng = random.Random(15)
for _ in range(400):
    a = [rng.randint(-6, 6) for _ in range(rng.randint(1, 9))]
    assert min_circular(a) == min_circular_brute(a)
    k = rng.randint(1, len(a))
    assert max_circular_at_most_k(a, k) == max_circular_at_most_k_brute(a, k)
    s, ln = best_segment(a)
    assert sum((a + a)[s:s + ln]) == oracle(a)
print("min circular, độ dài <= k (hàng đợi đơn điệu), trả đoạn tốt nhất: khớp vét cạn trên 400 mảng")
assert max_circular_at_most_k([5, -3, 5], 2) == 10 and max_circular_at_most_k([5, -3, 5], 1) == 5

min circular, độ dài <= k (hàng đợi đơn điệu), trả đoạn tốt nhất: khớp vét cạn trên 400 mảng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đoạn vòng | `(start, length)` | `O(n²)` | `O(1)` | oracle |
| 2. Kadane hai chiều | `max(max_sub, total − min_sub)` | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Tiền tố + hàng đợi đơn điệu | cửa sổ `n` trên mảng nhân đôi | `O(n)` | `O(n)` | khái quát độ dài tối đa `k` |
| 4. Tiền tố + hậu tố cực đại | hậu tố `j` + tiền tố `≤ j−1` | `O(n)` | `O(n)` | không nhánh toàn âm |

Cả bốn khớp oracle mọi-đoạn-vòng trên 8 001 mảng nhỏ và khớp nhau trên `n = 3·10^4` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().maxSubarraySumCircular([3, -1, 2, -1]), Solution().maxSubarraySumCircular([-5, -1, -3]), Solution().maxSubarraySumCircular([4, -9, 4]), min_circular([4, -9, 4]))

4 -1 8 -9


### Bài 2 (tính tay)

`nums = [8, -8, 9, -9, 10, -11, 12]`. (a) `total`? (b) `kadane_max`? (c) `kadane_min`? (d) Đáp án vòng? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện Kadane hai chiều:

In [30]:
def kadane2_todo(nums):
    total = cur_max = cur_min = 0
    best_max = best_min = nums[0]
    for x in nums:
        total += x
        pass            # TODO 1: cập nhật cur_max, best_max (Kadane max)
        pass            # TODO 2: cập nhật cur_min, best_min (Kadane min)
    pass                # TODO 3: toàn âm -> best_max; ngược lại max(best_max, total - best_min)


# Khi làm xong (nhớ thêm câu lệnh return ở TODO 3), bỏ comment để kiểm tra:
# assert [kadane2_todo(a) for a, _ in examples] == [3, 10, -2]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **tiền tố cực đại + hậu tố**:

In [31]:
class MySolution:
    def maxSubarraySumCircular(self, nums):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().maxSubarraySumCircular(a) == w for a, w in examples)

### Bài 5 (mini challenge): Maximum Sum of Two Non-Overlapping Subarrays (1031)

Cho `nums` và `firstLen`, `secondLen`, tìm tổng lớn nhất của hai đoạn con **không giao nhau** có độ dài đúng `firstLen` và `secondLen` (đoạn nào đứng trước cũng được). Viết `max_sum_two_no_overlap(nums, L, M)` trong `O(n)` và kiểm tra bằng vét cạn.

In [32]:
def max_sum_two_no_overlap(nums, L, M):
    pass            # TODO


# Khi làm xong:
# assert max_sum_two_no_overlap([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2) == 20

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `max(Kadane, total − min_sub)`, toàn âm thì Kadane.
- Mức 2: `[3,-1,2,-1]`: `total = 3`, `min_sub = −1`, `3 − (−1) = 4`; Kadane `4`.
- Mức 3: kết quả `4 -1 8 -9`: `[4,-9,4]` → `8` (đoạn vòng `[4, 4]`); tổng nhỏ nhất của đoạn vòng là `−9` (đoạn `[-9]`).

**Bài 2**

- Mức 1: `total = 8 − 8 + 9 − 9 + 10 − 11 + 12 = 11`.
- Mức 2: `kadane_max`: các đoạn dương `[12]`, `[10, −11, 12] = 11`, `[8,−8,9,−9,10,−11,12] = 11`...
- Mức 3: `kadane_min = −11`; vòng `11 − (−11) = 22`; đáp án `max(kadane_max, 22) = 22`.

**Bài 3**

- Mức 1: `cur_max = max(x, cur_max + x)`.
- Mức 2: `cur_min = min(x, cur_min + x)`.
- Mức 3: `return best_max if best_max < 0 else max(best_max, total - best_min)`.

**Bài 4**

- Mức 1: `best_pre[i]` = max tiền tố độ dài `1..i+1`.
- Mức 2: duyệt `j` từ `n-1` về `1`, cộng dồn hậu tố.
- Mức 3: `max(Kadane, suf + best_pre[j-1])`.

**Bài 5**

- Mức 1: tiền tố tổng, tổng cửa sổ `L`, `M`.
- Mức 2: với mỗi vị trí đoạn thứ hai, ghép với tổng cửa sổ `L` tốt nhất bên trái hoặc bên phải.
- Mức 3: chạy hai lần (đổi vai `L`, `M`) lấy max.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().maxSubarraySumCircular([3, -1, 2, -1]), Solution().maxSubarraySumCircular([-5, -1, -3]), Solution().maxSubarraySumCircular([4, -9, 4]), min_circular([4, -9, 4]))
assert vals == (4, -1, 8, -9) == (oracle([3, -1, 2, -1]), oracle([-5, -1, -3]), oracle([4, -9, 4]), min_circular_brute([4, -9, 4]))
print("Bài 1 OK:", vals)

Bài 1 OK: (4, -1, 8, -9)


### Bài 2

(a) `total = 11`. (b) `kadane_max = 12`. (c) `kadane_min = −11`. (d) vòng `11 − (−11) = 22`; đáp án `22`.

In [34]:
a = [8, -8, 9, -9, 10, -11, 12]
assert sum(a) == 11 and kadane_max(a) == 12 and kadane_min(a) == -11
assert Solution().maxSubarraySumCircular(a) == 22 == oracle(a)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def kadane2_answer(nums):
    total = cur_max = cur_min = 0
    best_max = best_min = nums[0]
    for x in nums:
        total += x
        cur_max = max(x, cur_max + x)
        best_max = max(best_max, cur_max)
        cur_min = min(x, cur_min + x)
        best_min = min(best_min, cur_min)
    return best_max if best_max < 0 else max(best_max, total - best_min)


assert [kadane2_answer(a) for a, _ in examples] == [3, 10, -2]
for a in tests[:300]:
    assert kadane2_answer(a) == oracle(a)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def maxSubarraySumCircular(self, nums):
        n = len(nums)
        best_pre, run, best = [0] * n, 0, float("-inf")
        for i, x in enumerate(nums):
            run += x
            best = max(best, run)
            best_pre[i] = best
        kad = cur = nums[0]
        for x in nums[1:]:
            cur = max(x, cur + x)
            kad = max(kad, cur)
        wrap, suf = float("-inf"), 0
        for j in range(n - 1, 0, -1):
            suf += nums[j]
            wrap = max(wrap, suf + best_pre[j - 1])
        return max(kad, wrap)


assert all(MySolutionAnswer().maxSubarraySumCircular(a) == w for a, w in examples)
for a in tests[:300]:
    assert MySolutionAnswer().maxSubarraySumCircular(a) == oracle(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def max_sum_two_no_overlap_answer(nums, L, M):
    n = len(nums)
    P = [0] + list(accumulate(nums))

    def one_order(a, b):                                  # đoạn dài a đứng trước, đoạn dài b đứng sau
        best_a, best = 0, 0
        for j in range(a + b, n + 1):
            best_a = max(best_a, P[j - b] - P[j - b - a])         # tổng đoạn a tốt nhất kết thúc trước vị trí j - b
            best = max(best, best_a + P[j] - P[j - b])
        return best

    return max(one_order(L, M), one_order(M, L))


def two_no_overlap_brute(nums, L, M):
    n = len(nums)
    best = 0
    for i in range(n - L + 1):
        for j in range(n - M + 1):
            if i + L <= j or j + M <= i:
                best = max(best, sum(nums[i:i + L]) + sum(nums[j:j + M]))
    return best


assert max_sum_two_no_overlap_answer([0, 6, 5, 2, 2, 5, 1, 9, 4], 1, 2) == 20
rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(0, 9) for _ in range(rng.randint(3, 12))]
    L = rng.randint(1, len(a) - 1)
    M = rng.randint(1, len(a) - L)
    assert max_sum_two_no_overlap_answer(a, L, M) == two_no_overlap_brute(a, L, M)
print("Bài 5 OK: khớp vét cạn")

Bài 5 OK: khớp vét cạn


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đoạn con vòng tròn → Kadane hai chiều, nhánh toàn âm):**

```python
class Solution:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        total = cur_max = cur_min = 0
        best_max = best_min = nums[0]
        for x in nums:
            total += x
            cur_max = max(x, cur_max + x); best_max = max(best_max, cur_max)
            cur_min = min(x, cur_min + x); best_min = min(best_min, cur_min)
        return best_max if best_max < 0 else max(best_max, total - best_min)
```

**Checklist trước khi submit:**

- [ ] Có cả **hai** trường hợp: không vòng (Kadane) và vòng (`total − min_sub`)?
- [ ] Trường hợp **toàn âm** (`best_max < 0`) trả `best_max`, không `0`?
- [ ] Không cho độ dài vượt `n` (không dùng phần tử hai lần)?
- [ ] Khởi tạo `best_max`, `best_min` bằng `nums[0]` (không `0`)?
- [ ] Đã thử `[1,-2,3,-2] → 3`, `[5,-3,5] → 10`, `[-3,-2,-3] → -2`, một phần tử, toàn dương, toàn `0`?
- [ ] Đúng tên `class Solution`, `maxSubarraySumCircular(self, nums)`, không in debug.